In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os


In [10]:
## CSV to pandas DataFrame Import##
from tabulate import tabulate
def csv_to_df(filename, filepath=None):
    """
    Extracts major and trace/REE element oxide data from a .csv and organizes it into a pandas DataFrame.
            - Each column is a different oxide/species/category.
            - Each sample is a row in the .csv file.
            - Each sample becomes a row in a pandas DataFrame.
            - Data can be filtered by sample name, oxide name, or special groups:
                - Major elements
                - Trace elements
                - REE elements

    ****At the moment, all data includes the LOI + sum values.

    Parameters:
        - filename: str - name of the .csv file
        - **folderpath: str, optional: path to folder if not sci-data

    Returns:
        pd.DataFrame: The imported data as a pandas DataFrame.
    """
    if filepath is None:
        script_dir = os.path.expanduser('~/PycharmProjects/scientific-coding-v1/sci-data/')
        filepath = os.path.join(script_dir, filename)

        df = pd.read_csv(filepath)
        column_names = list(df.columns)

        print(f"Columns:{column_names}")
        print(f"\n .csv file successfully imported.")
    return df

def csv_to_samples_list(filename, filepath=None, index_col=0, include_name=True, transpose_if_needed=True):
    """
    Read a CSV where oxide names are expected to be the row index (vertical) and samples are columns (horizontal).
    Produces a list-of-lists where each sub-list is one sample: [sample_name, val1, val2, ...] if include_name True,
    otherwise [val1, val2, ...]. Non-numeric or missing entries are coerced to NaN so plotting libraries ignore them.

    Returns:
        samples_list: list of lists
        oxides_order: list of oxide names (order of values in each sample sub-list)
        df_numeric: pandas DataFrame with oxides as index and samples as columns (numeric values, NaN for invalid)
    """
    import re

    if filepath is None:
        script_dir = os.path.expanduser('~/PycharmProjects/scientific-coding-v1/sci-data/')
        filepath = os.path.join(script_dir, filename)

    # Read CSV using the provided index column (defaults to first column being oxide names)
    df = pd.read_csv(filepath, index_col=index_col)

    # Heuristic to detect orientation: compute fraction of numeric values in columns and in index
    def numeric_fraction_series(s):
        coerced = pd.to_numeric(s, errors='coerce')
        return coerced.notna().mean()

    # Fraction of numeric values across columns (average per column)
    col_fractions = []
    for col in df.columns:
        col_fractions.append(numeric_fraction_series(df[col]))
    avg_col_numeric_frac = float(np.mean(col_fractions)) if col_fractions else 0.0

    # Fraction of numeric values in the index
    try:
        index_series = pd.Series(list(df.index))
        idx_numeric_frac = numeric_fraction_series(index_series)
    except Exception:
        idx_numeric_frac = 0.0

    # If columns are NOT numeric but the index looks numeric, then the CSV was probably transposed -> transpose
    if transpose_if_needed and (avg_col_numeric_frac < 0.5 and idx_numeric_frac > 0.5):
        df = df.T

    # Coerce all dataframe values to numeric, leaving non-numeric as NaN
    df_numeric = df.apply(lambda col: pd.to_numeric(col, errors='coerce'))

    return samples_list, oxides_order, df_numeric

    print(f"Imported {len(samples_list)} samples with {len(oxides_order)} oxides (non-numeric entries -> NaN.")

            samples_list.append(values)
        else:
samples_list.append([sample_name] + values)
        if include_name:
        values = list(df_numeric[sample_name].values)
    for sample_name in df_numeric.columns:
    samples_list = []

    oxides_order = list(df_numeric.index)

    df_numeric.index = df_numeric.index.map(lambda x: str(x))
    # Ensure index (oxides) are strings


IndentationError: unindent does not match any outer indentation level (<string>, line 84)

In [3]:
## First I'm going to create a function that will plot a single diagram based on a specific x and y axis input. 
def harker_diagram(plotLength, plotWidth, title, x, xerr, xlabel, y, yerr, ylabel, pointlabels, colorlist):   
    """
    Plots a Harker diagram with error bars for the specified x and y axes.

    Parameters:
        - plotLength: int - length of the plot
        - plotWidth: int - width of the plot
        - title: str - title of the plot
        - x: str - x-axis label
        - xlabel: str - x-axis label
        - y: str - y-axis label
        - ylabel: str - y-axis label
        - df: pd.DataFrame - DataFrame containing the data to plot
        - datarange: index - start:end of rows (samples) to plot
        - error: name of row that contains error measurements 
        
    """
    fig, ax = plt.subplots(figsize=(plotLength, plotWidth))
    fig.suptitle(title, fontsize=20)
    ax.errorbar(x, y, xerr=xerr, yerr=yerr, color=colorlist, fmt='o', ecolor=colorlist, capsize=1.5, alpha=0.7, markersize=4)
    ax.set_xlabel(xlabel, fontsize=13)
    ax.set_ylabel(ylabel, fontsize=13)
    
    for i, name in enumerate(pointlabels):
        ax.annotate(name, (x.iloc[i], y.iloc[i]), fontsize=10, ha='center', va='baseline',alpha=0.4 )
    
    # x-axis and labels
    return fig, ax